# From raw Reddit data to a housing dataset
*Group 20 · Dutch Housing Blame · notebook 01 (Adrian)*

Our research question is who people blame for the Dutch housing crisis online, and whether the Dutch-language
community (r/thenetherlands) blames different actors than the English-language one (r/Netherlands).
Before anyone can annotate or compare anything, we need a dataset that only contains housing discussions,
does not contain usernames, and is small enough to share with the rest of the group. That is what this notebook does.

The order below is the order I actually worked in: load everything, throw out what we can't use, remove usernames,
find the housing comments, add context and language, and finally save it all for notebooks 02–04.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 20
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
ANNOTATION = ROOT / "annotation"
FIGURES = ROOT / "figures"
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

In [ ]:
import gc
import hashlib
import html
import io
import os
import re
import zipfile

import zstandard
from dotenv import load_dotenv
from lingua import Language, LanguageDetectorBuilder

SUBREDDITS = ["thenetherlands", "Netherlands"]
DATA_PROC.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(exist_ok=True)

## 1. Where the data comes from
We collected the data with `collect.py` from [Arctic Shift](https://github.com/ArthurHeitmann/arctic_shift), an archive
of public Reddit posts and comments. My first plan was to use its keyword search to only download housing comments,
but on two subreddits this big the search kept timing out. So instead we downloaded *everything* from both subreddits
between 1 January 2024 and 31 October 2026 and do the filtering ourselves.

The script works week by week (148 weeks), asks for the posts or comments of one week page by page, and saves each
finished week as one compressed file. That way the 12-hour crawl could be stopped and restarted without losing anything.
It sends at most one request per second, waits when the server says so, and two weeks that failed the first time were
simply fetched again afterwards.

To check nothing went missing I compared our counts with the counts Arctic Shift publishes itself:

| | collected | Arctic Shift | coverage |
|---|---|---|---|
| r/thenetherlands posts | 38,913 | 38,931 | 99.95% |
| r/Netherlands posts | 79,747 | 79,782 | 99.96% |
| r/thenetherlands comments | 1,419,203 | 1,419,457 | 99.98% |
| r/Netherlands comments | 1,701,447 | 1,701,326 | 100.0% |

That is close enough to call the dataset complete.

## 2. Loading 3 million comments
The raw files are compressed JSON lines, one file per week. Reading one file is easy; the problem is memory. Kept as
normal Python strings, 3 million comments take several gigabytes, so I convert the text columns to pyarrow strings
*per file*, before putting everything together.

In [ ]:
POST_COLS = ["id", "author", "created_utc", "subreddit", "score", "title", "selftext", "num_comments"]
COMMENT_COLS = ["id", "author", "created_utc", "subreddit", "score", "body", "link_id", "parent_id"]
TEXT_COLS = ["id", "author", "subreddit", "title", "selftext", "body", "link_id", "parent_id"]


def read_week(path):
    with open(path, "rb") as f:
        lines = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(f), encoding="utf-8")
        return [json.loads(line) for line in lines]


def load_kind(kind, columns):
    frames = []
    for sub in SUBREDDITS:
        files = sorted((DATA_RAW / kind / sub).glob("*.jsonl.zst"))
        print(f"{kind}/{sub}: {len(files)} files")
        for path in files:
            df = pd.DataFrame(read_week(path), columns=columns)
            # pyarrow strings use much less memory than Python objects
            frames.append(df.astype({c: "string[pyarrow]" for c in columns if c in TEXT_COLS}))
    return pd.concat(frames, ignore_index=True)

In [ ]:
posts_raw = load_kind("posts", POST_COLS)
comments_raw = load_kind("comments", COMMENT_COLS)
print(f"comments_raw: {comments_raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")

About 1 GB for all comments, which is fine. A quick check that the numbers are exactly what the crawler saved:

In [ ]:
raw_counts = {(kind, sub): int((df["subreddit"] == sub).sum())
              for kind, df in [("posts", posts_raw), ("comments", comments_raw)]
              for sub in SUBREDDITS}
raw_counts

In [ ]:
# sanity check
EXPECTED = {("posts", "thenetherlands"): 38913, ("posts", "Netherlands"): 79747,
            ("comments", "thenetherlands"): 1419203, ("comments", "Netherlands"): 1701447}
assert raw_counts == EXPECTED, raw_counts
print("ok")

## 3. Cleaning: what we can't use
Not every comment is usable. Some are deleted by their author, some are removed by moderators (or Reddit), a few are
empty, and some are written by bots like AutoModerator. I count them per subreddit before dropping them, because the
difference between the two communities turned out to be interesting in itself.

In [ ]:
BOT = r"(?i)^automoderator$|bot$"


def drop_reason(df):
    text = df["body"].fillna("")
    reason = pd.Series(pd.NA, index=df.index, dtype="string")
    reason[text.str.strip() == ""] = "empty"
    reason[text == "[deleted]"] = "deleted"
    reason[text == "[removed]"] = "removed"
    reason[df["author"].fillna("").str.contains(BOT)] = "bot"
    return reason


comments_raw["drop_reason"] = drop_reason(comments_raw)
dropped = comments_raw.groupby("subreddit")["drop_reason"].value_counts().unstack(fill_value=0)
dropped["total"] = comments_raw.groupby("subreddit").size()
dropped["kept"] = dropped["total"] - dropped[["bot", "deleted", "empty", "removed"]].sum(axis=1)
dropped["dropped_share"] = (1 - dropped["kept"] / dropped["total"]).round(3)
dropped = dropped.reset_index()
dropped

This was one of the first surprises: r/thenetherlands loses **9.5%** of its comments and r/Netherlands only **2.3%**,
and almost all of it is content that was *removed*, not deleted by the author (8.8% vs. 1.9%). So r/thenetherlands is
moderated much more strictly. That matters for us: if removed comments are more often angry or blaming comments, the
blame in r/thenetherlands will be under-represented. I'll come back to this in the limitations.

Comments whose *account* was deleted but whose text is still there are kept. They just won't get a user id later.

Next, the text itself. Reddit comments are full of links, markdown and HTML codes, and a keyword inside a URL
(`.../huurwoning-amsterdam`) should not make a comment count as housing. I wrote a small cleaning function and tested it
on a few annoying cases first. It also replaces usernames that people mention in their comments (`u/...`), since those
would otherwise slip through the pseudonymisation.

In [ ]:
MD_LINK = re.compile(r"\[([^\]]*)\]\([^)]*\)")
URL = re.compile(r"https?://\S+|www\.\S+")
USER = re.compile(r"(?<![\w/])/?u/[A-Za-z0-9_-]+")
MARKDOWN = re.compile(r"[*_~`#^]+|^\s*>", re.M)
ESCAPED = re.compile(r"\\([^\w\s])")


def clean_text(text):
    text = MD_LINK.sub(r"\1", text)
    text = URL.sub(" ", text)
    text = html.unescape(text)
    text = ESCAPED.sub(r"\1", text)
    text = USER.sub("u/[user]", text)
    text = MARKDOWN.sub(" ", text)
    return " ".join(text.split())


tests = {
    "Zie [dit artikel](https://nos.nl/x) over **huurprijzen**": "Zie dit artikel over huurprijzen",
    "&gt; landlords &amp; investors\n\nexactly": "landlords & investors exactly",
    "\>eigen woning forfait weg": "eigen woning forfait weg",
    "ask u/some_user or /u/Other-1 about https://example.com/a?b=c": "ask u/[user] or u/[user] about",
}
for raw, expected in tests.items():
    assert clean_text(raw) == expected, (clean_text(raw), expected)
print("clean_text ok")

In [ ]:
comments = comments_raw[comments_raw["drop_reason"].isna()].copy()
comments["text"] = comments["body"].map(clean_text).astype("string[pyarrow]")
comments["n_words"] = comments["text"].str.count(r"\S+").astype(int)

posts = posts_raw.copy()
posts["title"] = posts["title"].fillna("").map(clean_text).astype("string[pyarrow]")
posts["removed"] = posts["selftext"].isin(["[removed]", "[deleted]"])
print(len(comments), "comments kept")

In [ ]:
# sanity check
assert "subreddit" in dropped.columns or "subreddit" in dropped.index.names
print(dropped)

## 4. Removing usernames
We promised in our plan that no usernames would leave my laptop, so every username becomes a hash: the first 16
characters of a SHA-256 hash of *salt + username*. The salt is a long random string stored in a `.env` file that is not
in git. Without it, anyone could hash a known username and look it up in our data; with it, the hashes can only be linked
back by someone who has the salt. The same username always gives the same hash, so we can still count how active a user
is or who replies to whom.

Hashing 3 million rows one by one is slow, so I hash every *unique* name once and map the result.

In [ ]:
load_dotenv(ROOT / ".env")
SALT = os.environ["PSEUDONYM_SALT"]


def pseudonymise(name):
    if pd.isna(name) or name == "[deleted]":
        return None
    return hashlib.sha256((SALT + name).encode("utf-8")).hexdigest()[:16]


def hash_column(names):
    # hash every unique name once instead of 3 million times
    lookup = {name: pseudonymise(name) for name in pd.unique(names)}
    return names.map(lookup).astype("string[pyarrow]")


assert pseudonymise("example") == pseudonymise("example") != pseudonymise("Example")
comments_raw["author_hash"] = hash_column(comments_raw["author"])
posts["author_hash"] = hash_column(posts["author"])
comments["author_hash"] = comments_raw.loc[comments.index, "author_hash"]

For the reply network in notebook 04 we also need to know *who* a comment is replying to. That person might have written
a non-housing comment or a comment that was later deleted, so the lookup has to use all comments and posts, which is
why I did this before throwing away the raw data.

In [ ]:
author_of = pd.concat([
    pd.Series(comments_raw["author_hash"].to_numpy(), index="t1_" + comments_raw["id"].astype(str)),
    pd.Series(posts["author_hash"].to_numpy(), index="t3_" + posts["id"].astype(str)),
])
comments["parent_author_hash"] = comments["parent_id"].astype(str).map(author_of).astype("string[pyarrow]")

comments = comments.drop(columns=["author", "body", "drop_reason"])
posts = posts.drop(columns=["author", "selftext"])
del comments_raw, posts_raw, author_of
gc.collect()
print("parent author known for", f"{comments['parent_author_hash'].notna().mean():.1%}", "of comments")

In [ ]:
# sanity check
assert "author" not in comments.columns
assert comments["author_hash"].dropna().str.len().eq(16).all()
print("ok")

## 5. Finding the housing comments
Now the core of this notebook. I use a bilingual keyword list: Dutch terms like *woningnood*, *huurwoning* or
*sociale huur*, and English ones like *housing*, *landlord* or *rental*. Dutch loves compound words (*huurwoningen*,
*studentenhuisvesting*), so a word only has to *start* with one of the terms.

One decision we had to make as a group was about *hypotheek* / *mortgage*. In a test on 12 sample weeks they were the most
common match in r/thenetherlands, about a third of all matches, but when I read them most were personal finance ("my
mortgage application"), not the housing debate. So they are out. The policy term *hypotheekrenteaftrek* (mortgage
interest deduction) stays in, because that one is part of the debate.

In [ ]:
NL_TERMS = [
    "woningnood", "woningcrisis", "wooncrisis", "woningmarkt", "woningtekort", "woningzoekend", "woningbouw",
    "huurwoning", "huurhuis", "huurhuizen", "huurprijs", "huurprijzen", "huurverhoging", "huurbescherming",
    "huurcommissie", "huurtoeslag", "huurder", "verhuurder", "huisjesmelker", "woningcorporatie", "sociale huur",
    "middenhuur", "vrije sector", "scheefwon", "overbied", "opkoopbescherming", "betaalbare huur",
    "starterswoning", "koopwoning", "huizenprijs", "huizenprijzen", "huizenmarkt", "hypotheekrenteaftrek",
    "kamernood", "studentenhuisvesting", "huisvesting",
]
EN_TERMS = [
    "housing", "rental", "rent price", "rent control", "rent increase", "landlord", "tenant", "house prices",
    "property prices", "outbid", "overbid", "mortgage interest deduction",
]
# \b only on the left: "huurwoning" also matches "huurwoningen", "landlord" also "landlords"
HOUSING_PATTERN = r"\b(?:" + "|".join(re.escape(t) for t in NL_TERMS + EN_TERMS) + ")"

A comment counts as housing-related if it contains a keyword itself, **or** if it was posted under a post whose title
contains one. The second rule catches replies like "they should just build more" that are clearly about housing without
using any of the words.

In [ ]:
comments["kw_match"] = comments["text"].str.contains(HOUSING_PATTERN, case=False, regex=True)
posts["housing_title"] = posts["title"].str.contains(HOUSING_PATTERN, case=False, regex=True)

housing_posts = set("t3_" + posts.loc[posts["housing_title"], "id"].astype(str))
comments["in_housing_thread"] = comments["link_id"].isin(housing_posts)

comments_housing = comments[comments["kw_match"] | comments["in_housing_thread"]].copy()
pd.DataFrame({
    "comments": comments.groupby("subreddit").size(),
    "kw_match": comments.groupby("subreddit")["kw_match"].sum(),
    "in_housing_thread": comments.groupby("subreddit")["in_housing_thread"].sum(),
    "housing (either)": comments_housing.groupby("subreddit").size(),
    "housing posts": posts.groupby("subreddit")["housing_title"].sum(),
})

Housing is clearly a bigger topic for the international community: a keyword shows up in 2.6% of r/Netherlands comments
against 1.6% in r/thenetherlands, and r/Netherlands has almost four times as many housing posts (2,170 vs. 595). In total
we end up with **113,592 housing comments**, about half of them only because they are in a housing thread.

A keyword list is only useful if it mostly catches the right things. To check that, I took 100 random keyword matches,
read every single one, and marked whether it is really about housing (1) or not (0).

In [ ]:
PRECISION_FILE = DATA_PROC / "precision_sample.csv"
if not PRECISION_FILE.exists():
    (comments_housing[comments_housing["kw_match"]]
     .sample(100, random_state=SEED)[["id", "subreddit", "text"]]
     .assign(is_housing="")
     .to_csv(PRECISION_FILE, index=False))
    print("Exported. Label the is_housing column in", PRECISION_FILE)

precision_sample = pd.read_csv(PRECISION_FILE)
if precision_sample["is_housing"].notna().all():
    print("precision overall:", round(precision_sample["is_housing"].mean(), 3))
    print(precision_sample.groupby("subreddit")["is_housing"].agg(["mean", "size"]))
else:
    print(precision_sample["is_housing"].isna().sum(), "rows still need a label")

In [ ]:
# sanity check
assert len(precision_sample) == 100 and precision_sample["is_housing"].notna().all()
print("precision:", precision_sample["is_housing"].mean())

**89 of the 100 are about housing (precision 0.89)**: 0.88 in r/Netherlands and 0.92 in r/thenetherlands. Reading the 11
misses was useful. Eight of them mention housing only in passing while talking about something else (cost of living,
politics, immigration), two are about *car* rental, and one is a joke about spending your rent money on goats.
The car rental ones could be filtered out, but it's 2% and I didn't think it was worth a more complicated pattern.
The "in passing" ones are not something keywords can fix, and the annotators will catch them anyway with the relevance
label. I'm happy with 0.89.

## 6. Context and language
When we annotate, a comment like "they should just build more" means nothing without knowing what it replies to.
So every housing comment gets the text of its parent: the comment it replies to, or the post title if it's a top-level
comment.

In [ ]:
text_of = pd.concat([
    pd.Series(comments["text"].to_numpy(), index="t1_" + comments["id"].astype(str)),
    pd.Series(posts["title"].to_numpy(), index="t3_" + posts["id"].astype(str)),
])
parent_text = comments_housing["parent_id"].astype(str).map(text_of)
missing = parent_text.isna().groupby(comments_housing["parent_id"].str[:3]).mean().rename("missing share")
comments_housing["parent_text"] = parent_text.fillna("").astype("string[pyarrow]")
missing

Only 0.3% of the parents are missing (deleted, or posted before 2024), so almost every comment has its context.

Language is the second thing. Our two communities are defined by language, but people don't always stick to it, so I
detect the language of every housing comment with `lingua`. My first try used lingua's confidence score for the whole
comment, but that hides mixing: a sentence that is half Dutch and half English still came out as 99% Dutch. So I let
lingua split each comment into language segments and count the words per language instead:

- fewer than 3 words: `other`, too short to say anything;
- at least 3 Dutch *and* 3 English words, and the smaller language is at least a quarter of the comment: `mixed`;
- otherwise whichever of Dutch or English has at least half of the words, and `other` if neither does.

I tried it on a few sentences first:

In [ ]:
detector = LanguageDetectorBuilder.from_languages(
    Language.DUTCH, Language.ENGLISH, Language.GERMAN, Language.FRENCH).build()


def lang_from_segments(segments):
    words = {}
    for s in segments:
        words[s.language] = words.get(s.language, 0) + s.word_count
    total = sum(words.values())
    nl, en = words.get(Language.DUTCH, 0), words.get(Language.ENGLISH, 0)
    if total < 3:
        return "other"
    if min(nl, en) >= 3 and min(nl, en) / total >= 0.25:
        return "mixed"
    if nl / total >= 0.5:
        return "nl"
    if en / total >= 0.5:
        return "en"
    return "other"


examples = {
    "Welke sociale huur? In de grote steden zit die potdicht.": "nl",
    "There is a housing crisis, so you would certainly struggle.": "en",
    "De huurprijzen zijn echt belachelijk. This landlord is insane and the government does nothing.": "mixed",
    "Dit is echt gezellig": "nl",
    "lol": "other",
}
for text, expected in examples.items():
    got = lang_from_segments(detector.detect_multiple_languages_of(text))
    print(f"{got:6} {expected:6} {text}")

In [ ]:
segments = detector.detect_multiple_languages_in_parallel_of(comments_housing["text"].tolist())
comments_housing["lang"] = [lang_from_segments(s) for s in segments]
del segments

And then just read a few random comments per language to see whether it makes sense:

In [ ]:
for lang in ["nl", "en", "mixed", "other"]:
    print(f"--- {lang}")
    for text in comments_housing.loc[comments_housing["lang"] == lang, "text"].sample(5, random_state=SEED):
        print("  ", text[:120])

In [ ]:
# sanity check
assert set(comments_housing["lang"].unique()) <= {"nl", "en", "mixed", "other"}
print(comments_housing.groupby("subreddit")["lang"].value_counts(normalize=True).round(3))

For housing, subreddit and language are almost the same thing: 94.8% of r/thenetherlands housing comments are Dutch and
95.6% of r/Netherlands ones are English. *Mixed* is rare (under 1%), and when I read those, quite a few are English
comments with a couple of Dutch housing words in them (*makelaar*, *huurcommissie*), so I see it more as an upper bound.

## 7. Who is talking?
Notebook 04 looks at people who are active in *both* communities, so I build one more table: for every user (hash) and
subreddit, how many comments they wrote, how many were about housing, and when they were active.

In [ ]:
comments["created"] = pd.to_datetime(comments["created_utc"], unit="s", utc=True)
comments_housing["created"] = pd.to_datetime(comments_housing["created_utc"], unit="s", utc=True)
posts["created"] = pd.to_datetime(posts["created_utc"], unit="s", utc=True)

author_activity = (comments.dropna(subset=["author_hash"])
                   .groupby(["author_hash", "subreddit"])
                   .agg(n_comments=("id", "size"), first_seen=("created", "min"), last_seen=("created", "max"))
                   .reset_index())
housing_counts = (comments_housing.dropna(subset=["author_hash"])
                  .groupby(["author_hash", "subreddit"]).size().rename("n_housing_comments").reset_index())
author_activity = author_activity.merge(housing_counts, on=["author_hash", "subreddit"], how="left")
author_activity["n_housing_comments"] = author_activity["n_housing_comments"].fillna(0).astype(int)
author_activity["author_hash"] = author_activity["author_hash"].astype(str)
author_activity.groupby("subreddit")["author_hash"].nunique()

76,531 users in r/thenetherlands and 167,863 in r/Netherlands, which matches the quick count I did on the raw data
earlier, so nothing got lost in the cleaning.

## 8. A first look at the data
Two figures for the data section of the paper. First, how much housing talk there is per month, both as a count and as a
share of everything that was posted, because a busy month for the whole subreddit is not the same as a month where
people talk more about housing.

In [ ]:
def per_month(df):
    month = df["created"].dt.tz_convert(None).dt.to_period("M").dt.to_timestamp()
    return df.groupby([month, "subreddit"]).size().unstack()


housing_month = per_month(comments_housing)
share_month = housing_month / per_month(comments)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
housing_month.plot(ax=axes[0])
axes[0].set_ylabel("housing comments")
(share_month * 100).plot(ax=axes[1], legend=False)
axes[1].set_ylabel("% of all comments")
axes[1].set_xlabel("")
axes[0].set_title("Housing comments per month")
fig.tight_layout()
fig.savefig(FIGURES / "01_housing_per_month.png", dpi=300, bbox_inches="tight")

r/Netherlands talks about housing more than r/thenetherlands almost every month, with peaks of up to 7–8% of all
comments in 2024. In 2025–2026 the gap gets smaller and in some months r/thenetherlands is even higher, which is something
to look at together with political events in notebook 03.

And the language mix, which shows the same thing as the numbers above:

In [ ]:
lang_mix = pd.crosstab(comments_housing["subreddit"], comments_housing["lang"], normalize="index") * 100
ax = lang_mix[["nl", "en", "mixed", "other"]].plot.barh(stacked=True, figsize=(8, 2.8))
ax.set_xlabel("% of housing comments")
ax.set_ylabel("")
ax.legend(ncol=4, bbox_to_anchor=(0.5, -0.35), loc="upper center")
ax.set_title("Language of housing comments")
plt.savefig(FIGURES / "01_language_mix.png", dpi=300, bbox_inches="tight")
lang_mix.round(1)

Finally, out of curiosity, which keywords actually do the work in each community:

In [ ]:
terms = (comments_housing.loc[comments_housing["kw_match"], ["subreddit", "text"]]
         .assign(term=lambda d: d["text"].str.lower().str.findall(HOUSING_PATTERN.replace("(?:", "(", 1)))
         .explode("term").dropna(subset=["term"]))
terms.groupby("subreddit")["term"].value_counts().groupby(level=0).head(10)

The difference already says something about the debate. In r/Netherlands it's *housing*, *landlord*, *rental* and
*tenant*, so the language of people who rent and deal with landlords. In r/thenetherlands it's *sociale huur*,
*woningbouw*, *woningmarkt* and *huizenprijzen*, which sounds more like the policy side: building, social housing and
the market.

## 9. Saving and sharing
Three files go to the rest of the group: the housing comments, the posts, and the user activity table. None of them
contain usernames. I also put them in one zip, because that's easier to share through SURFdrive than three separate
files. The texts themselves can still be searched on Reddit, so the zip stays within the group.

In [ ]:
HOUSING_COLS = ["id", "link_id", "parent_id", "subreddit", "created", "author_hash", "parent_author_hash",
                "score", "text", "parent_text", "kw_match", "in_housing_thread", "lang", "n_words"]
POST_OUT = ["id", "subreddit", "created", "author_hash", "title", "score", "num_comments", "removed", "housing_title"]

comments_housing[HOUSING_COLS].to_parquet(DATA_PROC / "comments_housing.parquet", index=False)
posts[POST_OUT].to_parquet(DATA_PROC / "posts.parquet", index=False)
author_activity.to_parquet(DATA_PROC / "author_activity.parquet", index=False)

# one zip to share with the group (pseudonymised, no usernames)
with zipfile.ZipFile(DATA_PROC / "group20_processed.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["comments_housing.parquet", "posts.parquet", "author_activity.parquet"]:
        z.write(DATA_PROC / name, arcname=name)
for f in sorted(DATA_PROC.glob("*")):
    print(f"{f.name:28} {f.stat().st_size / 1e6:7.1f} MB")

In [ ]:
# sanity check
REQUIRED = ["id", "link_id", "parent_id", "subreddit", "created", "author_hash", "parent_author_hash",
            "score", "text", "parent_text", "kw_match", "in_housing_thread", "lang", "n_words"]
df = pd.read_parquet(DATA_PROC / "comments_housing.parquet")
missing = set(REQUIRED) - set(df.columns)
assert not missing, missing
assert not {"author", "parent_author"} & set(df.columns)
for f in ["posts.parquet", "author_activity.parquet"]:
    assert (DATA_PROC / f).exists(), f
print(len(df), "housing comments, ok")

## What this means for the rest of the project
- We have **113,592 housing comments** from 2024 to October 2026, with a keyword filter that is right about 9 out of 10
  times. That is plenty for a 300-comment annotation sample.
- r/thenetherlands removes far more content than r/Netherlands (8.8% vs. 1.9%). This goes into the limitations.
- Housing is a bigger topic in the international community, and the two communities already use different words for it.
- Subreddit and language are almost the same split for housing comments, but since many users are active in both
  communities (notebook 04), we should still compare by language as a robustness check.